[<< Sommaire QC](../README.md) | [Precedent : QC-Py-Cloud-06-VolTargeting <<](./QC-Py-Cloud-06-VolTargeting.ipynb)

# QC-Py-Cloud-07 — Temporal CNN Direction Prediction

**Module** : Hands-On AI Trading, Ch.6 — Applied Machine Learning, Example 14

**Objectif** : Utiliser un reseau de neurones convolutif temporel (Temporal CNN) pour predire la direction future des prix (hausse/baisse/stationnaire) sur les actions de l'univers QQQ.

**Approche cloud-native** : L'algorithme est execute sur QuantConnect Cloud. Les resultats sont presentes ci-dessous.

> **Note de conception** : Ce notebook est un descriptif pedagogique. Le code executable se trouve dans le projet QuantConnect Cloud correspondant (voir le lien dans le notebook). L'absence d'outputs est intentionnelle.


## 1. Concept : Temporal CNN pour la prediction de direction

Le Temporal CNN utilise des couches de convolution 1D (Conv1D) pour extraire des caracteristiques temporelles a partir de donnees OHLCV historiques. Contrairement aux modeles de series temporelles classiques (ARIMA, HMM), le CNN apprend automatiquement les motifs pertinents.

### Architecture du modele

```
Input: (15 timesteps, 5 features) [OHLCV]
  |
Conv1D(30 filters, kernel=4, relu)  -- extraction de caracteristiques
  |
Split en 3 regions temporelles :  |  court terme  |  moyen terme  |  long terme  |
  |                                |               |               |
Conv1D(1 filter, kernel=1, relu)   |  Conv1D(...)  |  Conv1D(...)  |
  |                                                                    |
Concatenate + Flatten                                                 |
  |
Dense(3, softmax)  -->  [P(hausse), P(baisse), P(stationnaire)]
```

### Decisions de conception

1. **Split temporel** : La sortie du Conv1D est divisee en 3 regions (court/moyen/long terme), chacune traitee par un Conv1D(1,1). Cela force le modele a apprendre des motifs a differentes echelles temporelles.

2. **Classification 3 classes** : Au lieu de predire un prix continu, le modele classe la direction (UP/DOWN/STATIONARY). C'est plus robuste que la regression car il n'a pas besoin de predire l'amplitude exacte.

3. **Entrainement hebdomadaire** : Le modele est reentraine chaque semaine avec les 500 dernieres barres quotidiennes. Cela permet au modele de s'adapter aux changements de regime.

## 2. Parametres de la strategie

| Parametre | Valeur | Justification |
|-----------|--------|---------------|
| `training_samples` | 500 barres | Environ 2 ans de donnees quotidiennes |
| `universe_size` | 3 actions | Top 3 ponderations du QQQ |
| Confidence threshold | 55% | Seuil minimum pour trader |
| `n_tsteps` | 15 jours | Fenetre d'entree du CNN |
| Rolling avg window | 5 jours | Fenetre de calcul du label (direction) |
| Stationary threshold | 0.01% | Seuil pour classifier un mouvement comme stationnaire |
| Epochs | 20 | Cycles d'entrainement par retrain |
| Rebalance | Hebdomadaire | Retrain + prediction chaque lundi |

## 3. Resultats QC Cloud

**Projet QC Cloud** : 29816576 (`HandsOn-Ex14-Temporal-CNN`)
**Periode** : 2018-12-31 au 2024-01-01 (5 ans)
**Capital initial** : $100,000

### Resultat principal (Ex14-Temporal-CNN-v1)

| Metrique | Valeur |
|----------|--------|
| **Sharpe Ratio** | **0.734** |
| **CAGR** | **20.51%** |
| **Max Drawdown** | **21.6%** |
| **Net Profit** | **+154.1%** |
| **Total Trades** | 594 |
| **Win Rate** | 1.29 |
| **Loss Rate** | -1.22 |
| **Alpha** | 0.11 |
| **Beta** | 0.264 |
| **PSR** | 28.701 |
| **Sortino Ratio** | N/A |
| **Treynor Ratio** | 0.517 |

## 4. Analyse des performances

### 4.1 Le meilleur Sharpe du catalogue

Avec un Sharpe de 0.734, le Temporal CNN est la meilleure strategie du catalogue Cloud, devant le Sector Rotation (0.614) et le Risk Parity Composite (0.558). Le CAGR de 20.51% est impressionnant, avec un MaxDD de seulement 21.6%.

### 4.2 Alpha significatif

L'alpha de 0.11 (11% annualise) est le plus eleve de toutes les strategies. Combine avec un beta de seulement 0.264, cela signifie que la strategie genere des rendements independants du mouvement du marche. C'est la marque d'un veritable edge predictif.

### 4.3 Probabilistic Sharpe Ratio (PSR) eleve

Le PSR de 28.7 est exceptionnellement eleve. Le PSR mesure la probabilite que le Sharpe observe soit superieur a un benchmark (généralement 0). Un PSR > 2.0 est considere comme significatif. A 28.7, il n'y a pratiquement aucun doute que la performance n'est pas due au hasard.

### 4.4 Caveats

Malgre les metriques impressionnantes, plusieurs reserves s'appliquent :

- **Univers restreint** : 3 actions seulement (top QQQ). La strategie est concentree sur un petit nombre de positions.
- **Entrainement inline** : Le CNN est entraine directement dans l'algorithme QC (TensorFlow/Keras). Les performances dependent de la qualite de l'entrainement sur un nombre limite d'epochs (20) et de donnees (500 barres).
- **Pas de validation croise** : Pas de split train/test ou de walk-forward explicite dans l'algorithme. Le reentrainement hebdomadaire sur les 500 dernieres barres est une forme implicite de validation, mais pas une garantie.
- **Surapprentissage potentiel** : Un CNN avec 30 filtres Conv1D sur 500 echantillons peut surapprendre, surtout avec 20 epochs sans early stopping.

## 5. Comparaison avec les autres strategies Cloud

| Strategie | Sharpe | CAGR | MaxDD | Edge |
|-----------|--------|------|-------|------|
| Risk Parity v4 (Cloud-01) | 0.278 | 6.17% | 20.4% | Allocation diversifiee |
| Sector Rotation v3 (Cloud-02) | 0.614 | 10.76% | 15.5% | Trend following multi-asset |
| Dual Momentum v2 (Cloud-03) | 0.392 | 8.79% | 23.6% | Momentum + univers diversifie |
| Mean Reversion v2 (Cloud-04) | 0.307 | 5.89% | 14.6% | RSI + regime filter |
| PCA Stat Arb (Cloud-06) | 0.399 | 12.65% | 31.8% | PCA + OLS residuals |
| **Temporal CNN (Cloud-07)** | **0.734** | **20.51%** | **21.6%** | **Conv1D direction prediction** |

### Observations

1. **Le Temporal CNN domine** : Meilleur Sharpe (0.734), meilleur CAGR (20.51%), avec un MaxDD raisonnable (21.6%). La strategie combine le meilleur des deux mondes : rendement eleve et risque controle.

2. **L'edge du deep learning** : Contrairement aux strategies basees sur des indicateurs techniques (RSI, SMA, MACD), le Temporal CNN apprend ses propres caracteristiques a partir des donnees brutes. C'est un avantage quand les motifs de marche changent au fil du temps.

3. **Beta faible (0.264)** : La strategie est largement decorrelee du marche. Le rendement ne depend pas de la direction generale du marche mais de la capacite predictive du modele.

4. **Reserve** : Le univers restreint (3 actions) et l'absence de validation croisee formelle rendent ce resultat preliminaire. Une validation walk-forward multi-seed serait necessaire pour confirmer.

## 6. Lecons principales

### 6.1 Le deep learning bat les indicateurs techniques

Le Temporal CNN (Sharpe 0.734) surpasse toutes les strategies basees sur des indicateurs techniques. La capacity du CNN a extraire des motifs non lineaires a partir des donnees brutes est un avantage decisif par rapport aux indicateurs fixes.

### 6.2 La classification bat la regression

Predire la direction (3 classes) plutot que le prix exact est plus robuste. Le modele n'a pas besoin de predire l'amplitude du mouvement, seulement sa direction. Cela reduit le bruit dans les signaux.

### 6.3 Le reentrainement est essentiel

Le modele est reentraine chaque semaine. Sans reentrainement, les performances se degraderaient rapidement car les motifs de marche evoluent. Le reentrainement hebdomadaire est un compromis entre la fraicheur du modele et le cout computationnel.

### 6.4 La confiance filtre le bruit

Le seuil de confiance a 55% elimine les predictions incertaines. Un modele qui ne trade que lorsqu'il est confiant reduit le nombre de trades mais ameliore la qualite moyenne.

## 7. Pour aller plus loin

1. **Walk-forward validation** : Implementer une validation walk-forward 5-fold avec 4 seeds pour confirmer que le Sharpe de 0.734 est robuste et non un artefact du seed.

2. **Early stopping** : Ajouter un callback d'early stopping dans l'entrainement Keras pour eviter le surapprentissage.

3. **Univers elargi** : Tester avec 5-10 actions au lieu de 3 pour verifier si l'edge est scalable.

4. **Architecture LSTM/Transformer** : Comparer le Temporal CNN avec un LSTM ou un Transformer temporel pour voir si une architecture plus recente ameliore les performances.

5. **Ensemble de modeles** : Combiner les predictions de plusieurs CNN entraines avec des seeds differents pour reduire la variance des predictions.

**Reference** : Broad, J. (2025) — *Hands-On AI Trading with Python, QuantConnect, and AWS*, Chapter 6, Example 14.